# LLM Labeling — Strategy Comparison

Tests three prompting strategies against the human majority-vote consensus on the same 200 sentences (100 ECB, 100 Fed) used in `labeling_agreement.ipynb`.

- **Strategy A**: Zero-shot, minimal prompt
- **Strategy B**: Zero-shot, role-based prompt
- **Strategy C**: Chained + role + few-shot (2-step: salience → direction)

Human majority vote (2-of-3) is the ground truth. Agreement is measured with Cohen's κ so results are directly comparable to human-vs-human baselines.

## 0. Setup

In [1]:
import json
import time
from itertools import combinations
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

DATA_DIR = Path("../data/csv")
CACHE_PATH = DATA_DIR / "llm_labels_cache.json"
OUTPUT_PATH = DATA_DIR / "llm_labels.csv"

BASE_FILES = {
    "ecb": DATA_DIR / "ecb_random_tolabel.csv",
    "fed": DATA_DIR / "fed_random_tolabel.csv",
}
LABEL_FILES = {
    "daniel": {"ecb": DATA_DIR / "ecb_random_tolabel_daniel.csv", "fed": DATA_DIR / "fed_random_tolabel_daniel.csv"},
    "eric":   {"ecb": DATA_DIR / "ecb_random_tolabel_eric.csv",   "fed": DATA_DIR / "fed_random_tolabel_eric.csv"},
    "sam":    {"ecb": DATA_DIR / "ecb_random_tolabel_sam.csv",    "fed": DATA_DIR / "fed_random_tolabel_sam.csv"},
}
LABELERS = ["daniel", "eric", "sam"]
VALID_LABELS = {"hawkish", "dovish", "neutral"}
MODEL = "gpt-4o"

client = OpenAI()

pd.set_option("display.max_colwidth", 120)

In [2]:
def detect_separator(path: Path) -> str:
    header = path.open("r", encoding="utf-8", errors="replace").readline()
    return ";" if header.count(";") > header.count(",") else ","


def read_csv_clean(path: Path) -> pd.DataFrame:
    sep = detect_separator(path)
    df = pd.read_csv(path, sep=sep, encoding="utf-8", engine="python")
    df = df.loc[:, ~df.columns.str.startswith("Unnamed:")].copy()
    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"], errors="coerce")
    return df


def cohens_kappa(left: pd.Series, right: pd.Series) -> float:
    paired = pd.DataFrame({"left": left, "right": right}).dropna()
    if paired.empty:
        return float("nan")
    agreement = (paired["left"] == paired["right"]).mean()
    confusion = pd.crosstab(paired["left"], paired["right"])
    total = confusion.to_numpy().sum()
    row_probs = confusion.sum(axis=1) / total
    col_probs = confusion.sum(axis=0) / total
    expected = row_probs.mul(col_probs, fill_value=0).sum()
    if expected == 1:
        return float("nan")
    return (agreement - expected) / (1 - expected)

## 1. Build ground truth (human majority vote)

In [3]:
# Load base sample
base_frames = []
for bank in ["ecb", "fed"]:
    df = read_csv_clean(BASE_FILES[bank]).copy()
    df["bank"] = bank
    base_frames.append(df)
base_sample = pd.concat(base_frames, ignore_index=True)

# Join human labels
agreement_df = base_sample.drop(columns=["label"], errors="ignore").copy()
for labeler in LABELERS:
    label_frames = []
    for bank in ["ecb", "fed"]:
        lf = read_csv_clean(LABEL_FILES[labeler][bank]).copy()
        col = f"{labeler}_label"
        lf[col] = lf["label"].astype("string").str.strip().str.lower()
        lf["bank"] = bank
        label_frames.append(lf[["bank", "sentence_id", col]])
    agreement_df = agreement_df.merge(pd.concat(label_frames), on=["bank", "sentence_id"], how="left")

label_cols = [f"{l}_label" for l in LABELERS]
complete = agreement_df.dropna(subset=label_cols).copy()

# Majority vote
complete["majority_label"] = complete[label_cols].apply(lambda row: row.mode().iloc[0], axis=1)
complete["is_tie"] = complete[label_cols].nunique(axis=1) == 3

print(f"Total sentences: {len(complete)}")
print(f"Three-way ties (excluded from kappa): {complete['is_tie'].sum()}")
complete.groupby("bank")["majority_label"].value_counts().unstack(fill_value=0)

Total sentences: 200
Three-way ties (excluded from kappa): 9


majority_label,dovish,hawkish,neutral
bank,,,
ecb,15,13,72
fed,13,8,79


## 2. Select bank-specific few-shot examples

Examples are drawn from sentences where all 3 human labelers agreed (consensus cases). These sentences are excluded from evaluation to prevent leakage.

In [4]:
consensus = complete[~complete["is_tie"] & (complete[label_cols].nunique(axis=1) == 1)].copy()

# Review consensus sentences per bank/label to pick examples manually below
for bank in ["ecb", "fed"]:
    print(f"\n=== {bank.upper()} consensus sentences ===")
    for label in ["hawkish", "dovish", "neutral"]:
        subset = consensus[(consensus["bank"] == bank) & (consensus["majority_label"] == label)]
        print(f"\n  {label} ({len(subset)} sentences):")
        for _, row in subset.head(6).iterrows():
            print(f"    [{row['sentence_id']}] {row['text'][:100]}")


=== ECB consensus sentences ===

  hawkish (5 sentences):
    [20220609-ecb_103] And then we go further because we take a third step along that journey to indicate what we will do b
    [20230914-ecb_3] We are determined to ensure that inflation returns to our two per cent medium-term target in a timel
    [20181213-ecb_129] But one certainly is the extent to which profits are being squeezed by these increasing wages and, b
    [20240606-ecb_68] We will keep policy rates sufficiently restrictive for as long as necessary to achieve this aim.
    [20190124-ecb_99] Now, about inflation, of course going back to the first – the other question I had, focused first of

  dovish (2 sentences):
    [20231026-ecb_68] At the same time, inflation dropped markedly in September, including due to strong base effects, and
    [20200312-ecb_13] Together with the substantial monetary policy stimulus already in place, these measures will support

  neutral (54 sentences):
    [20240912-ecb_122] And of c

In [5]:
# Curated few-shot examples — 2 per class per bank, chosen from consensus above.
# Edit these after reviewing the output above. Prefer short, unambiguous sentences.
# sentence_id is used to exclude these from evaluation.

FEW_SHOT_EXAMPLES = {
    "ecb": {
        "neutral": [
            # Replace with actual sentence_id and text from the consensus output above
            {"sentence_id": "FILL_ME", "text": "FILL_ME"},
            {"sentence_id": "FILL_ME", "text": "FILL_ME"},
        ],
        "hawkish": [
            {"sentence_id": "FILL_ME", "text": "FILL_ME"},
            {"sentence_id": "FILL_ME", "text": "FILL_ME"},
        ],
        "dovish": [
            {"sentence_id": "FILL_ME", "text": "FILL_ME"},
            {"sentence_id": "FILL_ME", "text": "FILL_ME"},
        ],
    },
    "fed": {
        "neutral": [
            {"sentence_id": "FILL_ME", "text": "FILL_ME"},
            {"sentence_id": "FILL_ME", "text": "FILL_ME"},
        ],
        "hawkish": [
            {"sentence_id": "FILL_ME", "text": "FILL_ME"},
            {"sentence_id": "FILL_ME", "text": "FILL_ME"},
        ],
        "dovish": [
            {"sentence_id": "FILL_ME", "text": "FILL_ME"},
            {"sentence_id": "FILL_ME", "text": "FILL_ME"},
        ],
    },
}

# Collect all few-shot sentence_ids for leakage exclusion
FEWSHOT_IDS = {
    ex["sentence_id"]
    for bank_examples in FEW_SHOT_EXAMPLES.values()
    for examples in bank_examples.values()
    for ex in examples
    if ex["sentence_id"] != "FILL_ME"
}
print(f"Few-shot example sentence_ids to exclude from evaluation: {len(FEWSHOT_IDS)}")

Few-shot example sentence_ids to exclude from evaluation: 0


## 3. Prompt templates

In [6]:
SYSTEM_A = """Classify the following central bank sentence as hawkish, dovish, or neutral.
Respond with exactly one word."""

SYSTEM_B = """You are a central bank economist specializing in monetary policy communication.
Classify the following sentence as hawkish (signaling tighter policy / higher rates),
dovish (signaling looser policy / lower rates), or neutral (no clear policy signal).
Respond with exactly one word."""


def build_system_c_step1(bank: str) -> str:
    examples = FEW_SHOT_EXAMPLES[bank]
    neutral_block = "\n".join(f'  "{ex["text"]}" → neutral' for ex in examples["neutral"])
    opinion_block = "\n".join(
        f'  "{ex["text"]}" → opinionated'
        for label in ["hawkish", "dovish"]
        for ex in examples[label]
    )
    return f"""You are a central bank economist. Does this sentence express a monetary policy stance,
or is it factual/procedural with no clear signal?
Respond with exactly one word: opinionated or neutral.

Examples:
{neutral_block}
{opinion_block}"""


def build_system_c_step2(bank: str) -> str:
    examples = FEW_SHOT_EXAMPLES[bank]
    hawkish_block = "\n".join(f'  "{ex["text"]}" → hawkish' for ex in examples["hawkish"])
    dovish_block = "\n".join(f'  "{ex["text"]}" → dovish' for ex in examples["dovish"])
    return f"""You are a central bank economist. Is this sentence hawkish (signaling tighter policy)
or dovish (signaling looser policy)?
Respond with exactly one word: hawkish or dovish.

Examples:
{hawkish_block}
{dovish_block}"""

## 4. API call infrastructure

In [7]:
def load_cache() -> dict:
    if CACHE_PATH.exists():
        return json.loads(CACHE_PATH.read_text(encoding="utf-8"))
    return {}


def save_cache(cache: dict) -> None:
    CACHE_PATH.write_text(json.dumps(cache, indent=2), encoding="utf-8")


def call_openai(system_prompt: str, user_prompt: str, cache: dict, cache_key: str) -> str:
    if cache_key in cache:
        return cache[cache_key]
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        max_tokens=10,
        temperature=0,
    )
    result = response.choices[0].message.content.strip().lower()
    cache[cache_key] = result
    time.sleep(0.5)
    return result


def label_strategy_a(sentence_id: str, text: str, bank: str, cache: dict) -> str:
    key = f"A::{sentence_id}"
    result = call_openai(SYSTEM_A, text, cache, key)
    return result if result in VALID_LABELS else "invalid"


def label_strategy_b(sentence_id: str, text: str, bank: str, cache: dict) -> str:
    key = f"B::{sentence_id}"
    result = call_openai(SYSTEM_B, text, cache, key)
    return result if result in VALID_LABELS else "invalid"


def label_strategy_c(sentence_id: str, text: str, bank: str, cache: dict) -> str:
    key1 = f"C1::{sentence_id}"
    step1 = call_openai(build_system_c_step1(bank), text, cache, key1)
    if step1 == "neutral":
        return "neutral"
    key2 = f"C2::{sentence_id}"
    step2 = call_openai(build_system_c_step2(bank), text, cache, key2)
    return step2 if step2 in {"hawkish", "dovish"} else "invalid"

## 5. Run all strategies

Results are cached to `data/csv/llm_labels_cache.json` — re-running this cell will not incur additional API charges for already-labeled sentences.

In [8]:
cache = load_cache()
results = []
total = len(complete)
SAVE_EVERY = 10

for i, (_, row) in enumerate(complete.iterrows(), 1):
    sid = row["sentence_id"]
    text = row["text"]
    bank = row["bank"]
    print(f"[{i}/{total}] {bank} | {sid}", end=" ", flush=True)

    print("A...", end=" ", flush=True)
    a = label_strategy_a(sid, text, bank, cache)
    print("B...", end=" ", flush=True)
    b = label_strategy_b(sid, text, bank, cache)
    print("C...", end=" ", flush=True)
    c = label_strategy_c(sid, text, bank, cache)
    print(f"→ A={a} B={b} C={c}")

    results.append({"bank": bank, "sentence_id": sid, "strategy_a": a, "strategy_b": b, "strategy_c": c})
    if i % SAVE_EVERY == 0 or i == total:
        save_cache(cache)

llm_df = pd.DataFrame(results)
print(f"\nDone. Labeled: {len(llm_df)} sentences")
print("Invalid responses:")
print(llm_df[["strategy_a", "strategy_b", "strategy_c"]].apply(lambda col: (col == "invalid").sum()))


[1/200] ecb | 20220609-ecb_103 A... B... C... → A=hawkish B=hawkish C=hawkish
[2/200] ecb | 20240912-ecb_122 A... B... C... → A=neutral B=neutral C=neutral
[3/200] ecb | 20241212-ecb_151 A... B... C... → A=dovish B=dovish C=dovish
[4/200] ecb | 20251030-ecb_113 A... B... C... → A=neutral B=neutral C=neutral
[5/200] ecb | 20150903-ecb_151 A... B... C... → A=neutral B=neutral C=neutral
[6/200] ecb | 20230727-ecb_229 A... B... C... → A=neutral B=neutral C=neutral
[7/200] ecb | 20250911-ecb_64 A... B... C... → A=neutral B=neutral C=neutral
[8/200] ecb | 20251030-ecb_68 A... B... C... → A=neutral B=neutral C=neutral
[9/200] ecb | 20200430-ecb_54 A... B... C... → A=neutral B=neutral C=neutral
[10/200] ecb | 20221215-ecb_123 A... B... C... → A=neutral B=neutral C=neutral
[11/200] ecb | 20210909-ecb_138 A... B... C... → A=neutral B=neutral C=neutral
[12/200] ecb | 20150603-ecb_117 A... B... C... → A=dovish B=dovish C=dovish
[13/200] ecb | 20230914-ecb_3 A... B... C... → A=hawkish B=hawkish C=h

In [9]:
# Merge LLM labels onto complete (which has majority_label, is_tie, text)
eval_df = complete.merge(llm_df, on=["bank", "sentence_id"], how="left")
eval_df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved to {OUTPUT_PATH}")
eval_df[["bank", "sentence_id", "majority_label", "strategy_a", "strategy_b", "strategy_c"]].head()

Saved to ..\data\csv\llm_labels.csv


,bank,sentence_id,majority_label,strategy_a,strategy_b,strategy_c
0,ecb,20220609-ecb_103,hawkish,hawkish,hawkish,hawkish
1,ecb,20240912-ecb_122,neutral,neutral,neutral,neutral
2,ecb,20241212-ecb_151,dovish,dovish,dovish,dovish
3,ecb,20251030-ecb_113,neutral,neutral,neutral,neutral
4,ecb,20150903-ecb_151,neutral,neutral,neutral,neutral


## 6. Agreement vs. human consensus

Each LLM strategy is treated as a labeler and compared against the human majority vote using Cohen's κ — the same metric used for human-vs-human comparison in `labeling_agreement.ipynb`.

**Exclusions:** three-way ties + few-shot leakage sentences.

In [10]:
# Human baseline: each human vs. majority vote (excluding ties)
human_eval = eval_df[~eval_df["is_tie"]].copy()

human_rows = []
for labeler in LABELERS:
    col = f"{labeler}_label"
    for bank_name in ["ecb", "fed", "overall"]:
        subset = human_eval if bank_name == "overall" else human_eval[human_eval["bank"] == bank_name]
        human_rows.append({
            "labeler": labeler,
            "bank": bank_name,
            "n": len(subset),
            "pct_agreement": (subset[col] == subset["majority_label"]).mean(),
            "cohens_kappa": cohens_kappa(subset[col], subset["majority_label"]),
        })

human_baseline_df = pd.DataFrame(human_rows)
print("Human vs. majority vote:")
human_baseline_df.style.format({"pct_agreement": "{:.1%}", "cohens_kappa": "{:.3f}"})

Human vs. majority vote:


,labeler,bank,n,pct_agreement,cohens_kappa
0,daniel,ecb,95,90.5%,0.757
1,daniel,fed,96,91.7%,0.753
2,daniel,overall,191,91.1%,0.755
3,eric,ecb,95,87.4%,0.691
4,eric,fed,96,86.5%,0.599
5,eric,overall,191,86.9%,0.651
6,sam,ecb,95,86.3%,0.671
7,sam,fed,96,95.8%,0.857
8,sam,overall,191,91.1%,0.750


In [11]:
# LLM strategies vs. majority vote
llm_eval = eval_df[~eval_df["is_tie"] & ~eval_df["sentence_id"].isin(FEWSHOT_IDS)].copy()
strategies = {"strategy_a": "A: zero-shot", "strategy_b": "B: role-based", "strategy_c": "C: chained+few-shot"}

llm_rows = []
for col, name in strategies.items():
    for bank_name in ["ecb", "fed", "overall"]:
        subset = llm_eval if bank_name == "overall" else llm_eval[llm_eval["bank"] == bank_name]
        valid = subset[subset[col] != "invalid"]
        llm_rows.append({
            "strategy": name,
            "bank": bank_name,
            "n": len(valid),
            "invalid": len(subset) - len(valid),
            "pct_agreement": (valid[col] == valid["majority_label"]).mean(),
            "cohens_kappa": cohens_kappa(valid[col], valid["majority_label"]),
        })

llm_results_df = pd.DataFrame(llm_rows)
print("LLM strategies vs. human majority vote:")
llm_results_df.style.format({"pct_agreement": "{:.1%}", "cohens_kappa": "{:.3f}"})

LLM strategies vs. human majority vote:


,strategy,bank,n,invalid,pct_agreement,cohens_kappa
0,A: zero-shot,ecb,95,0,76.8%,0.443
1,A: zero-shot,fed,96,0,88.5%,0.661
2,A: zero-shot,overall,191,0,82.7%,0.543
3,B: role-based,ecb,95,0,80.0%,0.469
4,B: role-based,fed,96,0,89.6%,0.643
5,B: role-based,overall,191,0,84.8%,0.547
6,C: chained+few-shot,ecb,95,0,69.5%,0.334
7,C: chained+few-shot,fed,96,0,63.5%,0.193
8,C: chained+few-shot,overall,191,0,66.5%,0.263


In [12]:
# Combined summary table (human baselines + LLM strategies, overall only)
human_overall = human_baseline_df[human_baseline_df["bank"] == "overall"][["labeler", "pct_agreement", "cohens_kappa"]].rename(columns={"labeler": "name"})
llm_overall = llm_results_df[llm_results_df["bank"] == "overall"][["strategy", "pct_agreement", "cohens_kappa"]].rename(columns={"strategy": "name"})

summary = pd.concat([human_overall, llm_overall], ignore_index=True)
print("=== Summary: agreement with human majority vote ===")
summary.style.format({"pct_agreement": "{:.1%}", "cohens_kappa": "{:.3f}"})

=== Summary: agreement with human majority vote ===


,name,pct_agreement,cohens_kappa
0,daniel,91.1%,0.755
1,eric,86.9%,0.651
2,sam,91.1%,0.750
3,A: zero-shot,82.7%,0.543
4,B: role-based,84.8%,0.547
5,C: chained+few-shot,66.5%,0.263


In [17]:
# Pairwise human κ as the fair baseline (each human vs. another human, no self-reference)
pairwise_rows = []
for left, right in combinations(LABELERS, 2):
    left_col = f"{left}_label"
    right_col = f"{right}_label"
    pair_df = human_eval.dropna(subset=[left_col, right_col])
    pairwise_rows.append({
        "name": f"{left} × {right}",
        "pct_agreement": (pair_df[left_col] == pair_df[right_col]).mean(),
        "cohens_kappa": cohens_kappa(pair_df[left_col], pair_df[right_col]),
    })

pairwise_df = pd.DataFrame(pairwise_rows)
avg_row = pd.DataFrame([{
    "name": "human avg (pairwise)",
    "pct_agreement": pairwise_df["pct_agreement"].mean(),
    "cohens_kappa": pairwise_df["cohens_kappa"].mean(),
}])

# LLM strategies A and B only
llm_ab = llm_results_df[
    (llm_results_df["bank"] == "overall") &
    (llm_results_df["strategy"].isin(["A: zero-shot", "B: role-based"]))
][["strategy", "pct_agreement", "cohens_kappa"]].rename(columns={"strategy": "name"})

fair_summary = pd.concat([pairwise_df, avg_row, llm_ab], ignore_index=True)
print("=== Fair comparison: pairwise human κ vs. LLM strategies ===")
fair_summary.style.format({"pct_agreement": "{:.1%}", "cohens_kappa": "{:.3f}"})


=== Fair comparison: pairwise human κ vs. LLM strategies ===


,name,pct_agreement,cohens_kappa
0,daniel × eric,78.0%,0.428
1,daniel × sam,82.2%,0.515
2,eric × sam,78.0%,0.419
3,human avg (pairwise),79.4%,0.454
4,A: zero-shot,82.7%,0.543
5,B: role-based,84.8%,0.547


In [19]:
# GPT-4o as a 4th labeler — pairwise κ with each human
llm_human_rows = []
for labeler in LABELERS:
    human_col = f"{labeler}_label"
    for strategy_col, strategy_name in [("strategy_a", "A: zero-shot"), ("strategy_b", "B: role-based")]:
        pair_df = human_eval[
            (human_eval[strategy_col] != "invalid") &
            human_eval[human_col].notna()
        ]
        llm_human_rows.append({
            "name": f"{strategy_name} × {labeler}",
            "pct_agreement": (pair_df[strategy_col] == pair_df[human_col]).mean(),
            "cohens_kappa": cohens_kappa(pair_df[strategy_col], pair_df[human_col]),
        })

llm_human_df = pd.DataFrame(llm_human_rows)

avg_rows = pd.DataFrame([
    {"name": "── human × human avg", "pct_agreement": pairwise_df["pct_agreement"].mean(), "cohens_kappa": pairwise_df["cohens_kappa"].mean()},
    {"name": "── A × human avg",     "pct_agreement": llm_human_df[llm_human_df["name"].str.startswith("A")]["pct_agreement"].mean(), "cohens_kappa": llm_human_df[llm_human_df["name"].str.startswith("A")]["cohens_kappa"].mean()},
    {"name": "── B × human avg",     "pct_agreement": llm_human_df[llm_human_df["name"].str.startswith("B")]["pct_agreement"].mean(), "cohens_kappa": llm_human_df[llm_human_df["name"].str.startswith("B")]["cohens_kappa"].mean()},
])

# human-human individual pairs from earlier cell
human_human_df = pairwise_df[["name", "pct_agreement", "cohens_kappa"]].copy()

comparison = pd.concat([human_human_df, avg_rows.iloc[[0]], llm_human_df, avg_rows.iloc[[1,2]]], ignore_index=True)
print("=== Pairwise κ: human × human vs. LLM × human ===")
comparison.style.format({"pct_agreement": "{:.1%}", "cohens_kappa": "{:.3f}"})


=== Pairwise κ: human × human vs. LLM × human ===


,name,pct_agreement,cohens_kappa
0,daniel × eric,78.0%,0.428
1,daniel × sam,82.2%,0.515
2,eric × sam,78.0%,0.419
3,── human × human avg,79.4%,0.454
4,A: zero-shot × daniel,82.2%,0.541
5,B: role-based × daniel,84.3%,0.547
6,A: zero-shot × eric,74.3%,0.356
7,B: role-based × eric,77.5%,0.371
8,A: zero-shot × sam,82.2%,0.533
9,B: role-based × sam,83.2%,0.506


## 7. Disagreement inspection

Where does Strategy C (best strategy) disagree with human consensus? Are they the same hard sentences where humans also disagreed?

In [13]:
c_disagreements = llm_eval[
    (llm_eval["strategy_c"] != llm_eval["majority_label"]) &
    (llm_eval["strategy_c"] != "invalid")
][["bank", "sentence_id", "majority_label", "strategy_c", *[f"{l}_label" for l in LABELERS], "text"]]

print(f"Strategy C disagreements with majority: {len(c_disagreements)} / {len(llm_eval)}")
c_disagreements

Strategy C disagreements with majority: 64 / 191


,bank,sentence_id,majority_label,strategy_c,daniel_label,eric_label,sam_label,text
8,ecb,20200430-ecb_54,hawkish,neutral,hawkish,hawkish,neutral,Demand for loans to households for house purchase increased less than in the previous quarter.
11,ecb,20150603-ecb_117,neutral,dovish,neutral,hawkish,neutral,"But we should not forget that the structural component of our unemployment is high, and was high even before the cri..."
13,ecb,20171214-ecb_189,neutral,dovish,neutral,neutral,neutral,We cannot go beyond that.
17,ecb,20231026-ecb_68,dovish,neutral,dovish,dovish,dovish,"At the same time, inflation dropped markedly in September, including due to strong base effects, and most measures o..."
18,ecb,20171026-ecb_135,neutral,dovish,neutral,neutral,neutral,That's where our monetary policy becomes essential and that's what our monetary policy has in a sense contributed to...
...,...,...,...,...,...,...,...,...
189,fed,20160921-fed_7,dovish,neutral,dovish,dovish,neutral,"Economic growth, which was subdued during the first half of the year, appears to have \r\npicked up."
190,fed,20220126-fed_48,hawkish,neutral,hawkish,hawkish,hawkish,"[In] these high-level principles, [we] clarify that the federal \r\nfunds rate is our primary means of adjusting mon..."
192,fed,20220504-fed_226,neutral,dovish,dovish,neutral,neutral,"So it’s a good time to be a worker looking to, you know, either change jobs or get a \r\nwage increase in your curre..."
193,fed,20211103-fed_8,dovish,neutral,dovish,dovish,neutral,"Economic activity expanded at a 6.5 percent pace in the first half of the year, reflecting \r\nprogress on vaccinati..."


In [14]:
# Are strategy C's hard cases the same ones humans struggled with?
human_disagree_ids = set(
    eval_df[~eval_df["is_tie"] & (eval_df[label_cols].nunique(axis=1) > 1)]["sentence_id"]
)
c_disagree_ids = set(c_disagreements["sentence_id"])

overlap = c_disagree_ids & human_disagree_ids
print(f"Strategy C disagreements: {len(c_disagree_ids)}")
print(f"Human disagreements (2-of-3 majority, not unanimous): {len(human_disagree_ids)}")
print(f"Overlap (same hard sentences): {len(overlap)} ({len(overlap)/len(c_disagree_ids):.0%} of C's errors)")

Strategy C disagreements: 64
Human disagreements (2-of-3 majority, not unanimous): 59
Overlap (same hard sentences): 25 (39% of C's errors)


In [15]:
# Label distribution per strategy vs. human majority
dist_rows = []
for col, name in {"majority_label": "Human majority", **{k: v for k, v in strategies.items()}}.items():
    counts = llm_eval[col].value_counts(normalize=True)
    dist_rows.append({
        "labeler": name,
        "hawkish": counts.get("hawkish", 0.0),
        "dovish": counts.get("dovish", 0.0),
        "neutral": counts.get("neutral", 0.0),
        "invalid": counts.get("invalid", 0.0),
    })

print("Label distribution — strategies vs. human majority:")
pd.DataFrame(dist_rows).set_index("labeler").style.format("{:.1%}")

Label distribution — strategies vs. human majority:


,hawkish,dovish,neutral,invalid
labeler,,,,
Human majority,11.0%,9.9%,79.1%,0.0%
A: zero-shot,13.1%,11.5%,75.4%,0.0%
B: role-based,8.4%,9.9%,81.7%,0.0%
C: chained+few-shot,9.9%,25.7%,64.4%,0.0%
